# Практическая работа №2

## Детекторы и дескрипторы

## Задание

### Цель

Знакомство с детекторами и дескрипторами ключевых точек, формирование навыков поисков и классификации ключевых точек на языке Python.

### Задачи

Выполнение практической работы направлено на исследование детекторов и дескрипторов ключевых точек.

## Ход работы

[Как установить cv2 с запатентованными детекторами](https://stackoverflow.com/questions/69993071/attributeerror-module-cv2-cv2-has-no-attribute-surf-create-2-module-cv2)

### Импорты и вспомогательные функции

### Подготовка окружения

Устанавливаю необходимые зависимости через `pip`, если их ещё нет в окружении.


In [ ]:
%pip install numpy scipy scikit-image matplotlib opencv-contrib-python


In [ ]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt

In [ ]:
def detect_keypoints(detector, image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    kp, des = detector.detectAndCompute(gray, None)
    img = cv2.drawKeypoints(image, kp, None, flags = cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
    plt.imshow(img)
    plt.title(f'{len(kp)} keypoints')
    plt.axis('off')
    plt.show()

def detect_keypoints_harris(image, block_size=3, ksize=3, k=0.04, threshold=0.01):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    harris_corners = cv2.cornerHarris(gray, block_size, ksize, k)
    harris_corners = cv2.dilate(harris_corners, None)
    img = image.copy()
    img[harris_corners > threshold * harris_corners.max()] = [255, 0, 255]
    num_keypoints = np.sum(harris_corners > threshold * harris_corners.max())
    plt.imshow(img)
    plt.title(f'{num_keypoints} keypoints')
    plt.axis('off')
    plt.show()

def augment_image(image):
    augmented_images = []

    translation_matrix = np.float32([[1, 0, 150], [0, 1, 90]])
    translated_image = cv2.warpAffine(image, translation_matrix, (image.shape[1], image.shape[0]))
    augmented_images.append(translated_image)

    center = (image.shape[1] // 2, image.shape[0] // 2)
    angle = 20
    scale = 0.9
    rotation_matrix = cv2.getRotationMatrix2D(center, angle, scale)
    rotated_image = cv2.warpAffine(image, rotation_matrix, (image.shape[1], image.shape[0]))
    augmented_images.append(rotated_image)

    scale_factor = 1.5
    scaled_image = cv2.resize(image, None, fx=scale_factor, fy=scale_factor, interpolation=cv2.INTER_LINEAR)
    augmented_images.append(scaled_image)

    brightness_matrix = np.ones(image.shape, dtype=np.uint8) * 50
    brightened_image = cv2.add(image, brightness_matrix)
    augmented_images.append(brightened_image)

    dimmed_image = cv2.subtract(image, brightness_matrix)
    augmented_images.append(dimmed_image)

    height, width = image.shape[:2]
    src_points = np.float32([[0, 0], [width, 0], [0, height], [width, height]])
    dst_points = np.float32([[0, 0], [width * 0.9, height * 0.1], [width * 0.1, height], [width, height]])
    perspective_matrix = cv2.getPerspectiveTransform(src_points, dst_points)
    perspective_image = cv2.warpPerspective(image, perspective_matrix, (width, height))
    augmented_images.append(perspective_image)

    horizontal_flip = cv2.flip(image, 1)
    augmented_images.append(horizontal_flip)

    vertical_flip = cv2.flip(image, 0)
    augmented_images.append(vertical_flip)

    return augmented_images

def match_keypoints(detector, base_image, images, desc_norm='HAMMING', match_frac=0.5, num_cols=2):
    if desc_norm == 'HAMMING':
        matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    else:
        matcher = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True)
    base_gray = cv2.cvtColor(base_image, cv2.COLOR_RGB2GRAY)
    base_keypoints, base_descriptors = detector.detectAndCompute(base_gray, None)

    matched_images = []
    for img in images:
        gray_img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        keypoints, descriptors = detector.detectAndCompute(gray_img, None)

        matches = matcher.match(base_descriptors, descriptors)
        matches = sorted(matches, key=lambda x: x.distance)

        matched_image = cv2.drawMatches(base_image, base_keypoints, img, keypoints, matches[:int(len(matches) * match_frac)], None, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
        matched_images.append(matched_image)

    num_rows = (len(matched_images) + num_cols - 1) // num_cols
    plt.figure(figsize=(7 * num_cols, 3 * num_rows))

    for i, matched_image in enumerate(matched_images):
        plt.subplot(num_rows, num_cols, i + 1)
        plt.imshow(matched_image)
        plt.axis('off')

    plt.tight_layout()
    plt.show()

def match_keypoints_harris(base_image, images, threshold=0.01, num_cols=2):
    base_gray = cv2.cvtColor(base_image, cv2.COLOR_RGB2GRAY)
    base_corners = cv2.cornerHarris(base_gray, blockSize=2, ksize=3, k=0.04)
    base_corners = cv2.dilate(base_corners, None)
    base_image_corners = base_image.copy()
    base_image_corners[base_corners > threshold * base_corners.max()] = [255, 0, 255]

    matched_images = [base_image_corners]
    for img in images:
        gray_img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        corners = cv2.cornerHarris(gray_img, blockSize=2, ksize=3, k=0.04)
        corners = cv2.dilate(corners, None)
        img_corners = img.copy()
        img_corners[corners > threshold * corners.max()] = [255, 0, 255]

        matched_images.append(img_corners)

    num_rows = (len(matched_images) + num_cols - 1) // num_cols
    plt.figure(figsize=(7 * num_cols, 3 * num_rows))

    for i, matched_image in enumerate(matched_images):
        plt.subplot(num_rows, num_cols, i + 1)
        plt.imshow(matched_image)
        plt.axis('off')

    plt.tight_layout()
    plt.show()

### Подготовка изображений для исследования

In [ ]:
ben_images = []
book_images = []
man_images = []

data_dir =  './data'
for filename in os.listdir(data_dir):
    image = cv2.imread(os.path.join(data_dir, filename), cv2.IMREAD_COLOR)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    if filename.startswith('big_ben'):
        ben_images.append(image)
    elif filename.startswith('book'):
        book_images.append(image)
    elif filename.startswith('pushnoi'):
        man_images.append(image)
    else:
        raise ValueError(f'Unknown image: {filename}')

ben_augmented = augment_image(ben_images[0])
book_augmented = augment_image(book_images[0])
man_augmented = augment_image(man_images[0])

In [ ]:
plt.imshow(ben_images[0])

### SIFT (Scale-Invariant Feature Transform)

Метод имеет высокую инвариантность.
Удается выполнять сопоставление с похожими объектами.
Присутствуют ошибки при сопоставлении с другими объектами.

In [ ]:
detector = cv2.xfeatures2d.SIFT_create()

detect_keypoints(detector, ben_images[0])
detect_keypoints(detector, book_images[0])
detect_keypoints(detector, man_images[0])

In [ ]:
match_keypoints(detector, ben_images[0], ben_augmented + ben_images[1:] + book_images + man_images, desc_norm='NORM_L2', match_frac=0.3, num_cols=2)

In [ ]:
match_keypoints(detector, book_images[0], book_augmented + book_images[1:] + ben_images + man_images, desc_norm='NORM_L2', match_frac=0.3, num_cols=2)

In [ ]:
match_keypoints(detector, man_images[0], man_augmented + man_images[1:] + ben_images + book_images, desc_norm='NORM_L2', match_frac=0.3, num_cols=2)

### SURF (Speed Up Robust Feature)

Метод имеет высокую инвариантность.
Удается выполнять сопоставление с похожими объектами.
Очень много ошибок при сопоставлении с другими объектами.

In [ ]:
try:
    detector = cv2.xfeatures2d.SURF_create()
except (AttributeError, cv2.error):
    print('SURF detector is unavailable in the current OpenCV build.')
else:
    detect_keypoints(detector, ben_images[0])
    detect_keypoints(detector, book_images[0])
    detect_keypoints(detector, man_images[0])


In [ ]:
match_keypoints(detector, ben_images[0], ben_augmented + ben_images[1:] + book_images + man_images, desc_norm='NORM_L2', match_frac=0.3, num_cols=2)

In [ ]:
match_keypoints(detector, book_images[0], book_augmented + book_images[1:] + ben_images + man_images, desc_norm='NORM_L2', match_frac=0.3, num_cols=2)

In [ ]:
match_keypoints(detector, man_images[0], man_augmented + man_images[1:] + ben_images + book_images, desc_norm='NORM_L2', match_frac=0.3, num_cols=2)

### BRISK (Binary Robust Invariant Scalable Keypoints)

Метод имеет высокую инвариантность, но при горизонтальном отражении присутствуют ошибки.
Хорошо подходит для сопоставления с похожими объектами.
Присутствуют ошибки при сопоставлении с другими объектами.

In [ ]:
detector = cv2.BRISK_create()

detect_keypoints(detector, ben_images[0])
detect_keypoints(detector, book_images[0])
detect_keypoints(detector, man_images[0])

In [ ]:
match_keypoints(detector, ben_images[0], ben_augmented + ben_images[1:] + book_images + man_images, match_frac=0.3, num_cols=2)

In [ ]:
match_keypoints(detector, book_images[0], book_augmented + book_images[1:] + ben_images + man_images, match_frac=0.3, num_cols=2)

In [ ]:
match_keypoints(detector, man_images[0], man_augmented + man_images[1:] + ben_images + book_images, match_frac=0.3, num_cols=2)

### ORB (Oriented FAST and rotated BRIEF)

Метод выделяет не так много ключевых точек.
Он имеет хорошую инвариантность кроме случаев отражения.
Для сопоставления с похожими объектами подходит не очень.
Присутствуют ошибки при сопоставлении с другими объектами.

In [ ]:
detector = cv2.ORB_create()

detect_keypoints(detector, ben_images[0])
detect_keypoints(detector, book_images[0])
detect_keypoints(detector, man_images[0])

In [ ]:
match_keypoints(detector, ben_images[0], ben_augmented + ben_images[1:] + book_images + man_images, match_frac=0.3, num_cols=2)

In [ ]:
match_keypoints(detector, book_images[0], book_augmented + book_images[1:] + ben_images + man_images, match_frac=0.3, num_cols=2)

In [ ]:
match_keypoints(detector, man_images[0], man_augmented + man_images[1:] + ben_images + book_images, match_frac=0.3, num_cols=2)

### Harris

Реализация метода не предполагает получение дексрипторов ключевых точек, поэтому выполнить оценку сопоставления невозможно.

In [ ]:
detect_keypoints_harris(ben_images[0], block_size=3, ksize=3, k=0.04, threshold=0.04)
detect_keypoints_harris(book_images[0], block_size=3, ksize=3, k=0.04, threshold=0.04)
detect_keypoints_harris(man_images[0], block_size=3, ksize=3, k=0.04, threshold=0.04)

In [ ]:
match_keypoints_harris(ben_images[0], ben_augmented + ben_images[1:] + book_images + man_images, threshold=0.04, num_cols=2)

In [ ]:
match_keypoints_harris(book_images[0], book_augmented + book_images[1:] + ben_images + man_images, threshold=0.04, num_cols=2)

In [ ]:
match_keypoints_harris(man_images[0], man_augmented + man_images[1:] + ben_images + book_images, threshold=0.04, num_cols=2)

## Вывод

Были рассмотрены различные методы обнаружения ключевых точек и вычисления их дескрипторов, и изучены способы их применения.